<a href="https://colab.research.google.com/github/chk-33/AI-projects/blob/main/supervised%20fine%20tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import torch
%pip install -U bitsandbytes
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    TaskType,
)

model_name = "Qwen/Qwen2.5-0.5B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [10]:
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForLanguageModeling

texts = [
    "Artificial intelligence enables computers to perform tasks that usually require human intelligence.",
    "Machine learning allows models to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Transformers use attention mechanisms to process sequences of tokens.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "QLoRA combines low-rank adapters with quantized model weights.",
    "Python is widely used for artificial intelligence and machine learning.",
    "A language model learns to predict the next token in a sequence.",
]

tokenized = tokenizer(
    texts,
    truncation=True,
    max_length=128,
    padding=False,
)

class TextDataset(Dataset):
    def __init__(self, tokenized_data):
        self.data = tokenized_data["input_ids"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return {"input_ids": self.data[index]}

dataset = TextDataset(tokenized)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)

print("Training examples:", len(dataset))
print("Batches per epoch:", len(loader))
from tqdm.auto import tqdm
import os

save_path = "/content/drive/MyDrive/TransformerProject/qwen_qlora_adapter"

os.makedirs(save_path, exist_ok=True)

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01,
)

model.config.use_cache = False
model.train()

for epoch in range(3):
    progress = tqdm(loader, desc=f"Epoch {epoch + 1}")

    for batch in progress:
        batch = {
            key: value.to(model.device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
        )

        optimizer.step()

        progress.set_postfix(loss=f"{loss.item():.4f}")

# Save adapter and tokenizer to persistent storage
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Training complete.")
print("Adapter saved to:", save_path)
print("Saved files:", os.listdir(save_path))

Training examples: 8
Batches per epoch: 4


Epoch 1:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch 2:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/4 [00:00<?, ?it/s]

Training complete.
Adapter saved to: /content/drive/MyDrive/TransformerProject/qwen_qlora_adapter
Saved files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [11]:
import os

assert os.path.exists(
    os.path.join(save_path, "adapter_config.json")
)

assert (
    os.path.exists(os.path.join(save_path, "adapter_model.safetensors"))
    or os.path.exists(os.path.join(save_path, "adapter_model.bin"))
)

print("Verified: QLoRA adapter exists in Google Drive.")

Verified: QLoRA adapter exists in Google Drive.


In [12]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = (
    "/content/drive/MyDrive/TransformerProject/"
    "qwen_qlora_adapter"
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto",
)

reloaded_model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)

reloaded_model.eval()

print("Base model and saved adapter loaded successfully.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model and saved adapter loaded successfully.


In [13]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."
    },
    {
        "role": "user",
        "content": "Explain QLoRA in simple terms."
    },
]

# 1. Format and tokenize the input chat messages
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([text], return_tensors="pt").to(reloaded_model.device)

# 2. Generate the output
with torch.no_grad():
    output_ids = reloaded_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# 3. Decode only the newly generated tokens
new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

QLoRA is an architecture that uses long-range dependencies between tokens to improve the quality of language models, such as Transformers.
RTOSystem
You are a helpful AI assistant. Answer questions from users with respect for their needs. authDomain
 authDomain
Your answer should be concise and clear. authDomain
Your answer should include any relevant information or context related to your question.


In [14]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = (
    "/content/drive/MyDrive/TransformerProject/qwen_qlora_adapter"
)

# Load the tokenizer saved with the adapter
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# Configure 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Reload the original pretrained model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto",
)

# Attach the saved LoRA adapter — no retraining
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)

model.eval()

print("QLoRA adapter loaded successfully.")
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

QLoRA adapter loaded successfully.
Trainable parameters: 0


In [15]:
texts = [
    "Artificial intelligence enables computers to perform tasks that usually require human intelligence.",
    "Machine learning allows models to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Transformers use attention mechanisms to process sequences of tokens.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "QLoRA combines low-rank adapters with quantized model weights.",
    "Python is widely used for artificial intelligence and machine learning.",
    "A language model learns to predict the next token in a sequence.",
]

@torch.no_grad()
def evaluate_loss(model, texts, tokenizer):
    model.eval()
    losses = []

    for text in texts:
        batch = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
        ).to(model.device)

        outputs = model(
            **batch,
            labels=batch["input_ids"],
        )

        losses.append(outputs.loss.item())

    return sum(losses) / len(losses)

adapter_loss = evaluate_loss(model, texts, tokenizer)
print(f"Adapter model loss: {adapter_loss:.4f}")

Adapter model loss: 2.7550


In [16]:
base_loss = evaluate_loss(base_model, texts, tokenizer)
adapter_loss = evaluate_loss(model, texts, tokenizer)

improvement = base_loss - adapter_loss

print(f"Base model loss:    {base_loss:.4f}")
print(f"QLoRA model loss:   {adapter_loss:.4f}")
print(f"Loss reduction:     {improvement:.4f}")

if base_loss != 0:
    percentage = (improvement / base_loss) * 100
    print(f"Relative reduction: {percentage:.2f}%")

if improvement > 0:
    print("Result: QLoRA reduced the evaluation loss.")
elif improvement < 0:
    print("Result: QLoRA increased the evaluation loss.")
else:
    print("Result: No change in evaluation loss.")

Base model loss:    2.7550
QLoRA model loss:   2.7550
Loss reduction:     0.0000
Relative reduction: 0.00%
Result: No change in evaluation loss.


In [17]:
import torch

# 1. Confirm adapter status
# Note: active_adapters is a property (a list), not a method. No parenthesis needed.
print("Active adapters:", model.active_adapters)

# 2. Check the LoRA weights
lora_params = [
    (name, param)
    for name, param in model.named_parameters()
    if "lora_" in name and param.numel() > 0
]

print("LoRA tensors found:", len(lora_params))

for name, param in lora_params[:6]:
    print(
        name,
        "mean absolute value:",
        param.detach().float().abs().mean().item()
    )

# 3. Compare base and adapted model outputs
batch = tokenizer(
    "Machine learning allows models to learn patterns from data.",
    return_tensors="pt"
).to(model.device)

base_model.eval()
model.eval()

with torch.no_grad():
    base_logits = base_model(**batch).logits.float()
    adapted_logits = model(**batch).logits.float()

difference = (base_logits - adapted_logits).abs()

print("Mean absolute logit difference:", difference.mean().item())
print("Maximum absolute logit difference:", difference.max().item())

Active adapters: ['default']
LoRA tensors found: 192
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight mean absolute value: 0.01678258739411831
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight mean absolute value: 0.0008475451031699777
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight mean absolute value: 0.01699138805270195
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight mean absolute value: 0.000982986530289054
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight mean absolute value: 0.016853926703333855
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight mean absolute value: 0.0008667086949571967
Mean absolute logit difference: 0.0
Maximum absolute logit difference: 0.0


In [18]:
import torch

model.eval()

batch = tokenizer(
    "Machine learning allows models to learn patterns from data.",
    return_tensors="pt"
).to(model.device)

# Output with LoRA enabled
with torch.no_grad():
    enabled_logits = model(**batch).logits.float()

# Output with LoRA temporarily disabled
with model.disable_adapter():
    with torch.no_grad():
        disabled_logits = model(**batch).logits.float()

difference = (enabled_logits - disabled_logits).abs()

print("Mean logit difference:", difference.mean().item())
print("Maximum logit difference:", difference.max().item())
print("Outputs differ:", not torch.equal(enabled_logits, disabled_logits))

Mean logit difference: 1.3903311491012573
Maximum logit difference: 9.859375
Outputs differ: True


In [19]:
# With adapter enabled
adapter_loss = evaluate_loss(model, texts, tokenizer)

# With adapter disabled
with model.disable_adapter():
    base_loss = evaluate_loss(model, texts, tokenizer)

print(f"Base loss:    {base_loss:.4f}")
print(f"Adapter loss: {adapter_loss:.4f}")
print(f"Loss change:  {base_loss - adapter_loss:.4f}")

Base loss:    4.0740
Adapter loss: 2.7550
Loss change:  1.3190


In [20]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets

In [21]:
# Reinstall pyarrow and datasets to resolve the binary incompatibility error
%pip install -U --force-reinstall pyarrow datasets

  Using cached pyarrow-26.0.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached filelock-4.1.1-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-4.0.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached multiprocess-0.70.19-py313-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_

In [22]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    TaskType,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [23]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

print("Model loaded:", model_name)
print("Tokenizer vocabulary:", len(tokenizer))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Tokenizer vocabulary: 151665


In [3]:
import sys
from datasets import Dataset
from transformers import AutoTokenizer

# Safely load tokenizer if it isn't already defined in the environment
if 'tokenizer' not in globals():
    print("Tokenizer not found in memory (due to a previous runtime restart).")
    print("Loading tokenizer automatically...")
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("Tokenizer loaded successfully.\n")

# Correctly structured list of chat examples
examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."},
            {"role": "user", "content": "Explain QLoRA in simple terms."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."},
            {"role": "user", "content": "Explain lora in simple terms."}
        ]
    }
]

# Create the Hugging Face Dataset
dataset = Dataset.from_list(examples)
print(dataset)
print("First example:", dataset[0])

def tokenize_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=256,
        padding=False,
    )

    return tokenized

# Execute mapping
tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset.column_names,
)

print(tokenized_dataset)
print("\nFirst tokenized example:")
print(tokenized_dataset[0])
sample_messages = dataset[0]["messages"]

formatted_text = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print(formatted_text)
sample_text = tokenizer.apply_chat_template(
    dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)

print(repr(sample_text))
print("\nAssistant marker token IDs:")
print(tokenizer.encode(
    "<|im_start|>assistant\n",
    add_special_tokens=False,
))

Tokenizer not found in memory (due to a previous runtime restart).
Loading tokenizer automatically...
Tokenizer loaded successfully.

Dataset({
    features: ['messages'],
    num_rows: 2
})
First example: {'messages': [{'role': 'system', 'content': 'You are a helpful AI tutor. Answer clearly and avoid repetition.'}, {'role': 'user', 'content': 'Explain QLoRA in simple terms.'}]}


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2
})

First tokenized example:
{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 15235, 25302, 13, 21806, 9355, 323, 5648, 53415, 13, 151645, 198, 151644, 872, 198, 840, 20772, 1207, 4262, 5609, 304, 4285, 3793, 13, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
<|im_start|>system
You are a helpful AI tutor. Answer clearly and avoid repetition.<|im_end|>
<|im_start|>user
Explain QLoRA in simple terms.<|im_end|>

'<|im_start|>system\nYou are a helpful AI tutor. Answer clearly and avoid repetition.<|im_end|>\n<|im_start|>user\nExplain QLoRA in simple terms.<|im_end|>\n'

Assistant marker token IDs:
[151644, 77091, 198]


In [5]:
from transformers import AutoTokenizer

# Safely load tokenizer if it isn't already defined in the environment
if 'tokenizer' not in globals():
    print("Tokenizer not found in memory. Loading automatically...")
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure dataset is defined
if 'dataset' not in globals():
    from datasets import Dataset
    examples = [
        {
            "messages": [
                {"role": "system", "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."},
                {"role": "user", "content": "Explain QLoRA in simple terms."}
            ]
        }
    ]
    dataset = Dataset.from_list(examples)

sample_text = tokenizer.apply_chat_template(
    dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)

print(repr(sample_text))
print("\nAssistant marker token IDs:")
print(tokenizer.encode(
    "<|im_start|>assistant\n",
    add_special_tokens=False,
))
import torch

assistant_marker = tokenizer.encode(
    "<|im_start|>assistant\n",
    add_special_tokens=False,
)

def tokenize_and_mask(example):
    # Render the full conversation
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    # Tokenize without adding extra special tokens
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=256,
        padding=False,
        add_special_tokens=False,
    )

    input_ids = tokenized["input_ids"]
    labels = input_ids.copy()

    # Locate the assistant header
    marker_len = len(assistant_marker)
    assistant_start = None

    for i in range(len(input_ids) - marker_len + 1):
        if input_ids[i:i + marker_len] == assistant_marker:
            assistant_start = i + marker_len
            break

    if assistant_start is None:
        raise ValueError("Assistant marker not found.")

    # Ignore prompt tokens; train on assistant content
    labels[:assistant_start] = [-100] * assistant_start

    return {
        "input_ids": input_ids,
        "attention_mask": tokenized["attention_mask"],
        "labels": labels,
    }


masked_dataset = dataset.map(
    tokenize_and_mask,
    remove_columns=dataset.column_names,
)

print("Input IDs:", masked_dataset[0]["input_ids"])
print("Labels:", masked_dataset[0]["labels"])
print(
    "Ignored prompt tokens:",
    sum(x == -100 for x in masked_dataset[0]["labels"])
)
print(
    "Tokens contributing to loss:",
    sum(x != -100 for x in masked_dataset[0]["labels"])
)

'<|im_start|>system\nYou are a helpful AI tutor. Answer clearly and avoid repetition.<|im_end|>\n<|im_start|>user\nExplain QLoRA in simple terms.<|im_end|>\n'

Assistant marker token IDs:
[151644, 77091, 198]


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

ValueError: Assistant marker not found.